In [ ]:
from a2a.types import AgentCapabilities, AgentCard, AgentSkill
from aap_core.agent import BaseAgent
from aap_core.orchestration import VotingAgent
from aap_core.types import AgentMessage, BaseLLMChain
from aap_langchain.chain import ChatCausalMultiTurnsChain
from langchain_ollama import ChatOllama

/data/agent_design_pattern/src/langchain/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
class Agent(BaseAgent):
    chain: BaseLLMChain

    def execute(self, message: AgentMessage, **kwargs) -> AgentMessage:
        self.state = "running"
        # Set the chain's name to match the agent's card name for proper response attribution
        self.chain.name = self.card.name
        message = self.chain.invoke(message, **kwargs)
        message.execution_result = "success"
        message.origin = self.card.name
        self.state = "idle"
        return message

In [3]:
# System prompts for different agents
ev_system_prompt = """You are an Electric Vehicle (EV) advocate who specializes in pure electric vehicles.
Your expertise includes battery technology, charging infrastructure, environmental impact, and total cost of ownership.
You strongly believe that pure electric vehicles are the future of transportation.
Provide detailed, well-reasoned arguments supporting EVs over hybrid vehicles.
"""

hybrid_system_prompt = """You are a Hybrid Vehicle advocate who specializes in hybrid electric vehicles (HEVs and PHEVs).
Your expertise includes fuel efficiency, range anxiety solutions, practicality for long distances, and transitional technology.
You believe hybrid vehicles offer the best practical solution for current transportation needs.
Provide detailed, well-reasoned arguments supporting hybrid vehicles over pure electric vehicles.
"""

user_prompt = """{query}

Please provide your analysis and recommendation based on your expertise.
Be specific about advantages, disadvantages, costs, and practical considerations.
"""

def state_callback(state: str):
    print(f"agent state: {state}")

In [4]:
# Create agent cards and skills
ev_skill = AgentSkill(
    id='ev-skill',
    name="EV advocate skill",
    description="Expert in pure electric vehicles",
    tags=['ev', 'electric']
)
ev_card = AgentCard(
    name="ev_advocate",
    description="Electric Vehicle advocate agent",
    skills=[ev_skill],
    capabilities=AgentCapabilities(),
    default_input_modes=['text'],
    default_output_modes=['text'],
    url="localhost",
    version="0.1.0"
)

hybrid_skill = AgentSkill(
    id='hybrid-skill',
    name="Hybrid advocate skill",
    description="Expert in hybrid electric vehicles",
    tags=['hybrid', 'hev', 'phev']
)
hybrid_card = AgentCard(
    name="hybrid_advocate",
    description="Hybrid Vehicle advocate agent",
    skills=[hybrid_skill],
    capabilities=AgentCapabilities(),
    default_input_modes=['text'],
    default_output_modes=['text'],
    url="localhost",
    version="0.1.0"
)

voting_skill = AgentSkill(
    id='voting-skill',
    name="Voting skill",
    description="Voting agent skill",
    tags=['voting']
)
voting_card = AgentCard(
    name="voting_agent",
    description="Voting agent that selects the best answer",
    skills=[voting_skill],
    capabilities=AgentCapabilities(),
    default_input_modes=['text'],
    default_output_modes=['text'],
    url="localhost",
    version="0.1.0"
)

In [5]:
# Setup models and chains - use separate model instances for parallel execution
ev_model = ChatOllama(model="qwen3:4b-thinking-2507-q4_K_M", base_url="localhost:11434", temperature=0.7)
hybrid_model = ChatOllama(model="qwen3:4b-thinking-2507-q4_K_M", base_url="localhost:11434", temperature=0.7)

ev_chain = ChatCausalMultiTurnsChain(
    model=ev_model,
    system_prompt=ev_system_prompt,
    user_prompt_template=user_prompt
)

hybrid_chain = ChatCausalMultiTurnsChain(
    model=hybrid_model,
    system_prompt=hybrid_system_prompt,
    user_prompt_template=user_prompt
)

ev_agent = Agent(chain=ev_chain, card=ev_card, state_change_callback=state_callback)
hybrid_agent = Agent(chain=hybrid_chain, card=hybrid_card, state_change_callback=state_callback)

In [7]:
# Create VotingAgent with majority_vote method using BLEU score
voting_agent_majority = VotingAgent(
    agents=[ev_agent, hybrid_agent],
    card=voting_card,
    voting_method="majority_vote",
    scorer="bleu",
    state_change_callback=state_callback
)

print("VotingAgent (majority_vote with BLEU) created successfully")
print(f"Voting method: {voting_agent_majority.voting_method}")
print(f"Scorer: {voting_agent_majority.scorer}")

VotingAgent (majority_vote with BLEU) created successfully
Voting method: majority_vote
Scorer: bleu


In [9]:
# Test VotingAgent with majority_vote (BLEU score)
# Since Ollama is single-threaded, we execute agents sequentially and collect responses

query = """Compare pure electric vehicles (EVs) and hybrid vehicles (HEVs/PHEVs) for a typical urban commuter in the United States.
Consider the following factors:
- Initial purchase cost and total cost of ownership
- Environmental impact (including battery production)
- Convenience and charging infrastructure
- Performance and driving experience
- Range and suitability for long-distance travel

Which would you recommend and why?"""

# Execute agents sequentially and collect responses
print("=== Executing Agents Sequentially ===")
ev_message = ev_agent.execute(AgentMessage(query=query))
print(f"EV Agent completed. Response from: {ev_message.responses[-1][0]}")

hybrid_message = hybrid_agent.execute(AgentMessage(query=query))
print(f"Hybrid Agent completed. Response from: {hybrid_message.responses[-1][0]}")

# Create a message with both responses for the VotingAgent
voting_input = AgentMessage(
    query=query,
    responses=[
        ev_message.responses[-1],
        hybrid_message.responses[-1]
    ]
)

print(f"\n=== Running VotingAgent (majority_vote with BLEU) ===")
voting_message = voting_agent_majority.execute(voting_input)

print(f"\nVoting completed. Total responses: {len(voting_message.responses)}")
print("\n" + "="*80)
for agent_name, response in voting_message.responses:
    print(f"\n[{agent_name}]:")
    print("-" * 80)
    print(response)
    print("="*80)

=== Executing Agents Sequentially ===
agent state: voting_agent:idle/((ev_advocate:running)-(hybrid_advocate:idle))
agent state: voting_agent:idle/((ev_advocate:idle)-(hybrid_advocate:idle))
EV Agent completed. Response from: ev_advocate
agent state: voting_agent:idle/((ev_advocate:idle)-(hybrid_advocate:running))
agent state: voting_agent:idle/((ev_advocate:idle)-(hybrid_advocate:idle))
Hybrid Agent completed. Response from: hybrid_advocate

=== Running VotingAgent (majority_vote with BLEU) ===
agent state: voting_agent:running/((ev_advocate:idle)-(hybrid_advocate:idle))
agent state: voting_agent:idle/((ev_advocate:idle)-(hybrid_advocate:idle))

Voting completed. Total responses: 3


[ev_advocate]:
--------------------------------------------------------------------------------
## Comprehensive Comparison: Pure EVs vs. Hybrids for the Typical US Urban Commuter

As a dedicated EV advocate with deep expertise in battery technology, charging infrastructure, environmental impact, and tota

In [10]:
# Create VotingAgent with llm_score method
# First, define a scorer function that converts LLM response to a score
def score_response(response: str) -> float:
    """Simple scoring function that rates response quality."""
    # Score based on response length and presence of key terms
    score = len(response) / 100.0  # Base score on length
    key_terms = ['recommend', 'advantage', 'disadvantage', 'cost', 'environment', 'convenience']
    for term in key_terms:
        if term.lower() in response.lower():
            score += 0.5
    return min(score, 10.0)  # Cap at 10

In [11]:
# Voting prompt for llm_score method
voting_prompt = """Evaluate the quality of the following response about vehicle recommendations.
Score it from 1 to 10 based on:
- Comprehensiveness of analysis
- Clarity of arguments
- Practical recommendations
- Consideration of multiple factors

Only output a number between 1 and 10."""

voting_agent_llm = VotingAgent(
    agents=[ev_agent, hybrid_agent],
    card=voting_card,
    voting_method="llm_score",
    scorer=score_response,
    voting_prompt=voting_prompt,
    state_change_callback=state_callback
)

print("VotingAgent (llm_score) created successfully")
print(f"Voting method: {voting_agent_llm.voting_method}")

VotingAgent (llm_score) created successfully
Voting method: llm_score


In [ ]:
# Test VotingAgent with llm_score method
query = """I'm a family of four living in suburban California. We need a vehicle that can:
- Handle daily school runs and commutes (about 30 miles per day)
- Accommodate road trips to the coast (200+ miles one way) a few times per year
- Be cost-effective for a middle-income family
- Have reasonable environmental impact

Should we choose a pure electric vehicle or a hybrid? Please provide detailed reasoning."""

# Execute agents sequentially and collect responses
print("=== Executing Agents Sequentially ===")
ev_message = ev_agent.execute(AgentMessage(query=query))
print(f"EV Agent completed. Response from: {ev_message.responses[-1][0]}")

hybrid_message = hybrid_agent.execute(AgentMessage(query=query))
print(f"Hybrid Agent completed. Response from: {hybrid_message.responses[-1][0]}")

# Create a message with both responses for the VotingAgent
voting_input = AgentMessage(
    query=query,
    responses=[
        ev_message.responses[-1],
        hybrid_message.responses[-1]
    ]
)

print(f"\n=== Running VotingAgent (llm_score) ===")
message = voting_agent_llm.execute(voting_input)

print(f"Total responses: {len(message.responses)}")
print("\n" + "="*80)
for agent_name, response in message.responses:
    print(f"\n[{agent_name}]:")
    print("-" * 80)
    print(response)
    print("="*80)

=== Executing Agents Sequentially ===
agent state: voting_agent:idle/((ev_advocate:running)-(hybrid_advocate:idle))
agent state: voting_agent:idle/((ev_advocate:idle)-(hybrid_advocate:idle))
EV Agent completed. Response from: ev_advocate
agent state: voting_agent:idle/((ev_advocate:idle)-(hybrid_advocate:running))


In [6]:
# Create VotingAgent with Rouge-L score
voting_agent_rouge = VotingAgent(
    agents=[ev_agent, hybrid_agent],
    card=voting_card,
    voting_method="majority_vote",
    scorer="rougeL",
    state_change_callback=state_callback
)

print("VotingAgent (majority_vote with Rouge-L) created successfully")
print(f"Voting method: {voting_agent_rouge.voting_method}")
print(f"Scorer: {voting_agent_rouge.scorer}")

VotingAgent (majority_vote with Rouge-L) created successfully
Voting method: majority_vote
Scorer: rougeL


In [7]:
# Test VotingAgent with Rouge-L score
query = """For a first-time car buyer in Texas with a budget of $35,000:
- Compare the total cost of ownership over 5 years for EV vs Hybrid
- Discuss maintenance requirements
- Evaluate Texas-specific factors (climate, charging infrastructure, incentives)
- Recommend the best option with justification"""

# Execute agents sequentially and collect responses
print("=== Executing Agents Sequentially ===")
ev_message = ev_agent.execute(AgentMessage(query=query))
print(f"EV Agent completed. Response from: {ev_message.responses[-1][0]}")

hybrid_message = hybrid_agent.execute(AgentMessage(query=query))
print(f"Hybrid Agent completed. Response from: {hybrid_message.responses[-1][0]}")

# Create a message with both responses for the VotingAgent
voting_input = AgentMessage(
    query=query,
    responses=[
        ev_message.responses[-1],
        hybrid_message.responses[-1]
    ]
)

print(f"\n=== Running VotingAgent (Rouge-L) ===")
message = voting_agent_rouge.execute(voting_input)

print(f"Total responses: {len(message.responses)}")
print("\n" + "="*80)
for agent_name, response in message.responses:
    print(f"\n[{agent_name}]:")
    print("-" * 80)
    print(response)
    print("="*80)

=== Executing Agents Sequentially ===
agent state: voting_agent:idle/((ev_advocate:running)-(hybrid_advocate:idle))
agent state: voting_agent:idle/((ev_advocate:idle)-(hybrid_advocate:idle))
EV Agent completed. Response from: ev_advocate
agent state: voting_agent:idle/((ev_advocate:idle)-(hybrid_advocate:running))
agent state: voting_agent:idle/((ev_advocate:idle)-(hybrid_advocate:idle))
Hybrid Agent completed. Response from: hybrid_advocate

=== Running VotingAgent (Rouge-L) ===
agent state: voting_agent:running/((ev_advocate:idle)-(hybrid_advocate:idle))
agent state: voting_agent:idle/((ev_advocate:idle)-(hybrid_advocate:idle))
Total responses: 3


[ev_advocate]:
--------------------------------------------------------------------------------
## Comprehensive Analysis: EV vs. Hybrid for a First-Time Texas Buyer ($35k Budget)

As a pure EV specialist with deep expertise in battery tech, charging infrastructure, and real-world cost analysis (especially in the U.S. Southwest), I've meti

In [8]:
print("\n" + "="*80)
print("VOTING AGENT EXAMPLES COMPLETED")
print("="*80)
print("\nDemonstrated VotingAgent with:")
print("1. majority_vote method using BLEU score")
print("2. llm_score method with custom scorer function")
print("3. majority_vote method using Rouge-L score")
print("\nAll examples compared pure electric vehicles vs hybrid vehicles.")


VOTING AGENT EXAMPLES COMPLETED

Demonstrated VotingAgent with:
1. majority_vote method using BLEU score
2. llm_score method with custom scorer function
3. majority_vote method using Rouge-L score

All examples compared pure electric vehicles vs hybrid vehicles.
